# Alphabet transformer: on-policy gradient vs tokenwise gradient

Same sequence objective as `notebook/test.ipynb`, with the tabular Markov policy
replaced by a causal transformer.

The student is a **2-layer** transformer of width **`d_model = 32`**. It generates
length-**32** strings over `a–z` (token 0 is a BOS input and is never emitted).
The teacher is a frozen transformer of the same shape with a different init.
The state at step \(t\) is the full prefix \(y_{<t}\).

$$
J(\theta)=\mathbb{E}_{y\sim\pi_\theta}\sum_{t=0}^{T-1}
D_\alpha\big(\pi_\theta(\cdot\mid y_{<t}),\,\mathrm{teacher}(\cdot\mid y_{<t})\big)
$$

\(D_\alpha\) matches `compute_self_distillation_loss` (imported from
`variance_study.py`):

| \(\alpha\) | divergence |
| --- | --- |
| `0` | forward KL \(\mathrm{KL}(\mathrm{teacher}\,\|\,\pi)\) |
| `1` | reverse KL \(\mathrm{KL}(\pi\,\|\,\mathrm{teacher})\) |
| else | generalized JSD (default `0.5`) |

Two gradients of this objective:

- **On-policy gradient.** Differentiate \(J\) directly. Each string is drawn from
  \(\pi_\theta(y)=\prod_t\pi_\theta(y_t\mid y_{<t})\), so the likelihood-ratio identity
  gives the exact gradient with no occupancy recursion:

$$
\nabla J=\mathbb{E}_{y\sim\pi_\theta}\Big[\sum_t\nabla D_\alpha(y_{<t})
+\big(R(y)-b\big)\,\nabla\log\pi_\theta(y)\Big],\qquad
R(y)=\sum_t D_\alpha(y_{<t}),\quad \log\pi_\theta(y)=\sum_t\log\pi_\theta(y_t\mid y_{<t})
$$

  It is the autograd of the per-string loss
  \(\sum_t D_\alpha(y_{<t})+\mathrm{sg}(R(y)-b)\log\pi_\theta(y)\). The return is the
  whole-string sum, and the log-likelihood is that of the whole sampled string. The
  scalar \(b\) is the mean held-out return. It changes no expectation, but without it
  \(R\approx 130\) makes the estimate useless. A large-batch average of this loss is
  the reference `g_star`.

  Causality allows reward-to-go: \(D_k\) does not depend on \(y_t\) for \(k\le t\), so
  \(\mathrm{sg}(G_t-b_t)\log\pi(y_t\mid y_{<t})\) with \(G_t=\sum_{k>t}D_k\) has the same
  expectation and lower variance. That row is `"reward-to-go held-out baseline"`; it is an estimator, not the
  definition.
  Section 2 checks that the two large-batch averages agree.

- **Tokenwise gradient.** Autograd of \(\sum_t D_\alpha(y_{<t})\) on the sampled
  prefixes. It drops the score term. This is the `"no PG term"` row in the
  credit-weight sections.

Init scale is `0.5` so both policies depend on context and the prefix distribution
moves with \(\theta\). Bump the knobs in the next code cell; a CPU run of the
defaults is about half a minute.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "verl").is_dir() else CWD.parent
TOY_DIR = ROOT / "tests" / "special_standalone" / "future_returns_variance"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(TOY_DIR))

from variance_study import (  # noqa: E402
    divergence_name,
    future_returns,
    remaining_weights,
    token_divergence,
)

LETTERS = tuple("abcdefghijklmnopqrstuvwxyz")
N_LETTERS = len(LETTERS)



class CausalBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True, bias=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        t = x.shape[1]
        causal = torch.ones(t, t, device=x.device, dtype=torch.bool).triu(1)
        attn, _ = self.attn(h, h, h, attn_mask=causal, need_weights=False)
        x = x + attn
        h = self.ln2(x)
        return x + self.fc2(F.gelu(self.fc1(h)))


class TinyTransformer(nn.Module):
    def __init__(self, n_letters: int, d_model: int, n_layers: int, n_heads: int, d_ff: int, max_len: int):
        super().__init__()
        self.embed = nn.Embedding(n_letters + 1, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList(CausalBlock(d_model, n_heads, d_ff) for _ in range(n_layers))
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_letters, bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        pos = torch.arange(idx.shape[1], device=idx.device)
        x = self.embed(idx) + self.pos(pos)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


def init_model(seed: int) -> TinyTransformer:
    g = torch.Generator().manual_seed(seed)
    model = TinyTransformer(N_LETTERS, D_MODEL, N_LAYERS, N_HEADS, D_FF, max_len=T)
    for p in model.parameters():
        if p.dim() >= 2:
            nn.init.normal_(p, mean=0.0, std=INIT_STD, generator=g)
    return model


def student_params(model: nn.Module) -> list[nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]


def flat_grad(grads) -> torch.Tensor:
    return torch.cat([g.reshape(-1) for g in grads])


def prefixes(letters: torch.Tensor) -> torch.Tensor:
    bos = torch.zeros(letters.shape[0], 1, dtype=torch.long, device=letters.device)
    if letters.shape[1] == 1:
        return bos
    return torch.cat([bos, letters[:, :-1] + 1], dim=1)


def decode(row: torch.Tensor) -> str:
    return "".join(LETTERS[int(i)] for i in row.tolist())


@torch.no_grad()
def rollout(model: TinyTransformer, n: int, length: int) -> torch.Tensor:
    device = next(model.parameters()).device
    idx = torch.zeros(n, 1, dtype=torch.long, device=device)
    chars = []
    for _ in range(length):
        probs = F.softmax(model(idx)[:, -1], dim=-1)
        chars.append(torch.multinomial(probs, 1))
        idx = torch.cat([idx, chars[-1] + 1], dim=1)
    return torch.cat(chars, dim=1)


def sequence_rewards(student, teacher, letters, alpha: float):
    idx = prefixes(letters)
    s_logp = F.log_softmax(student(idx), dim=-1)
    with torch.no_grad():
        t_logp = F.log_softmax(teacher(idx), dim=-1)
    rewards = token_divergence(s_logp, t_logp.exp(), t_logp, alpha)
    logp_a = s_logp.gather(-1, letters.unsqueeze(-1)).squeeze(-1)
    return rewards, logp_a


def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()


def batch_grad(student, teacher, letters, weight_fn, alpha: float) -> torch.Tensor:
    rewards, logp_a = sequence_rewards(student, teacher, letters, alpha)
    weight = weight_fn(rewards.detach())
    loss = (rewards.sum(-1) + (weight * logp_a).sum(-1)).mean()
    return flat_grad(torch.autograd.grad(loss, student_params(student)))


def per_sample_grads(student, teacher, letters, weight_fn, alpha: float) -> torch.Tensor:
    rewards, logp_a = sequence_rewards(student, teacher, letters, alpha)
    weight = weight_fn(rewards.detach())
    per_seq = rewards.sum(-1) + (weight * logp_a).sum(-1)
    params = student_params(student)
    rows = [
        flat_grad(torch.autograd.grad(per_seq[i], params, retain_graph=True))
        for i in range(per_seq.shape[0])
    ]
    
    return torch.stack(rows)


def grad_metrics(g: torch.Tensor, g_star: torch.Tensor) -> dict:
    mean = g.mean(0)
    noise = (g - mean).pow(2).sum(-1).mean().sqrt()
    snr = (mean.norm() / noise).item() if float(noise) > 0 else float("nan")
    return {"snr": snr, "cos": cosine(mean, g_star), "mean_norm": mean.norm().item(), "noise": noise.item()}


def group_key(name: str) -> str:
    if name.startswith("embed") or name.startswith("pos"):
        return "embed"
    if name.startswith("blocks."):
        return f"layer {name.split('.')[1]}"
    return "head"


def parameter_spans(model: nn.Module):
    spans: dict[str, list[tuple[int, int]]] = {}
    order: list[str] = []
    offset = 0
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        key = group_key(name)
        if key not in spans:
            order.append(key)
            spans[key] = []
        spans[key].append((offset, offset + p.numel()))
        offset += p.numel()
    return order, spans


def take(flat: torch.Tensor, spans: list[tuple[int, int]]) -> torch.Tensor:
    return torch.cat([flat[a:b] for a, b in spans])


def credit_weight_fns(length: int):
    rem = remaining_weights(length)

    def as_rem(g):
        return rem.to(device=g.device, dtype=g.dtype)

    return {
        "raw G_t": lambda r: future_returns(r),
        "G_t - batch_mean_t": lambda r: future_returns(r) - future_returns(r).mean(0, keepdim=True),
        "G(r - batch_mean_t)": lambda r: future_returns(r - r.mean(0, keepdim=True)),
        "G_t / remaining": lambda r: future_returns(r) / as_rem(r),
        "(G_t - batch_mean_t)/remaining": lambda r: (
            future_returns(r) - future_returns(r).mean(0, keepdim=True)
        ) / as_rem(r),
        "G(r - batch_mean_t)/remaining": lambda r: future_returns(r - r.mean(0, keepdim=True)) / as_rem(r),
        "G_t / (T-1)": lambda r: future_returns(r) / (r.shape[1] - 1),
        "G(r - batch_mean_t)/(T-1)": lambda r: future_returns(r - r.mean(0, keepdim=True)) / (r.shape[1] - 1),
        "no PG term": lambda r: torch.zeros_like(r),
    }


def heldout_weight_fn(baseline_r: torch.Tensor):
    baseline_G = future_returns(baseline_r.reshape(1, -1)).reshape(-1)

    def fn(rewards: torch.Tensor) -> torch.Tensor:
        return future_returns(rewards) - baseline_G.to(device=rewards.device, dtype=rewards.dtype)

    return fn


def trajectory_weight_fn(baseline_R: float):
    def fn(rewards: torch.Tensor) -> torch.Tensor:
        return (rewards.sum(-1, keepdim=True) - baseline_R).expand_as(rewards)

    return fn


plt.rcParams.update({"figure.figsize": (9.5, 4.6), "axes.grid": True, "grid.alpha": 0.3})
print("repo", ROOT)


## Knobs

Depth 2, width 32, generated length 32. `N_TRUE` on-policy sequences are
sampled once; the whole-trajectory loss is averaged over them and backpropagated in one
pass to form the reference gradient.


In [ ]:
D_MODEL = 32
N_LAYERS = 2
N_HEADS = 4
D_FF = 128
T = 32
ALPHA = 0.0  # 0=forward KL, 1=reverse KL, 0.5=JSD
SEED = 0
INIT_STD = 0.5
N_BASE = 256
N_TRUE = 8192
N_ANAL = 128
N_METRICS = 32
N_POOL = 256
REPS = 4
BATCH_SIZES = [8, 16, 32, 64, 128]

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student = init_model(SEED).to(DEVICE)
teacher = init_model(SEED + 1).to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad_(False)
student.train()
n_params = sum(p.numel() for p in student.parameters())
print(
    f"device={DEVICE}  divergence={divergence_name(ALPHA)}  "
    f"depth={N_LAYERS}  d_model={D_MODEL}  T={T}  params={n_params}"
)


## 1. On-policy alphabet rollouts

Held-out sequences fix the baselines: the scalar \(b\) (mean return) and the
per-position \(b_t\) (future sum of the mean divergence). The curve is the mean token divergence
along the string; the band is one standard deviation across sequences.


In [ ]:
baseline_letters = rollout(student, N_BASE, T)
with torch.no_grad():
    rewards_base, _ = sequence_rewards(student, teacher, baseline_letters, ALPHA)
baseline_r = rewards_base.mean(0)
rtg_w = heldout_weight_fn(baseline_r)
seq_return = rewards_base.sum(-1)
traj_w = trajectory_weight_fn(seq_return.mean().item())

print(
    f"mean token D={baseline_r.mean().item():.3f}  "
    f"mean sequence return={seq_return.mean().item():.2f}  "
    f"std={seq_return.std().item():.2f}"
)
for i in range(8):
    print(f"  {decode(baseline_letters[i])}   sum_t D={seq_return[i].item():.2f}")

fig, ax = plt.subplots(figsize=(9.5, 3.6))
pos = torch.arange(T)
mean = rewards_base.mean(0)
std = rewards_base.std(0)
ax.plot(pos.cpu(), mean.cpu(), marker="o", ms=3)
ax.fill_between(pos, (mean - std).cpu(), (mean + std).cpu(), alpha=0.2)
ax.set_xlabel("position t")
ax.set_ylabel(r"$D_\alpha$")
ax.set_title(f"On-policy token divergence  {divergence_name(ALPHA)}")
fig.tight_layout()
plt.show()


## 2. True on-policy gradient vs tokenwise gradient

The reference `g_star` is a direct Monte Carlo gradient: sample `N_TRUE` strings from
the student, average the whole-trajectory loss
\(\sum_t D_t+\mathrm{sg}(R-b)\log\pi(y)\) over them, and backpropagate once.
Split-half cosine compares the gradients of the two halves of that sample (each
`N_TRUE / 2` strings); the full-sample reference is less noisy than that.
The reward-to-go loss on a fresh `N_TRUE` sample is the check.
Both are unbiased for \(\nabla J\), so their cosine should approach 1.

On one shared batch the on-policy gradient (the whole-trajectory loss) splits into the
tokenwise piece \(\sum_t \nabla D_t\) and the score-function piece
\((R-b)\sum_t \nabla\log\pi_t\).
Per-layer cosines use that same split. Per-position curves check that the
position grads add back to the full vector.

The finite-difference check perturbs the largest tokenwise coordinate on 8 strings.


In [ ]:
true_letters = rollout(student, N_TRUE, T)
g_star = batch_grad(student, teacher, true_letters, traj_w, ALPHA)
half = N_TRUE // 2
ref_cos = cosine(
    batch_grad(student, teacher, true_letters[:half], traj_w, ALPHA),
    batch_grad(student, teacher, true_letters[half:], traj_w, ALPHA),
)
g_rtg = batch_grad(student, teacher, rollout(student, N_TRUE, T), rtg_w, ALPHA)
print(
    f"||reference||={g_star.norm():.3f}  split-half cosine={ref_cos:.3f}  "
    f"sequences={N_TRUE}"
)
print(f"cos(whole-trajectory reference, reward-to-go average)={cosine(g_star, g_rtg):.3f}")

letters = rollout(student, N_ANAL, T)
g_on = batch_grad(student, teacher, letters, traj_w, ALPHA)
g_tok = batch_grad(student, teacher, letters, lambda r: torch.zeros_like(r), ALPHA)
g_pg = g_on - g_tok

piece_df = pd.DataFrame([
    {
        "piece": name,
        "norm": g.norm().item(),
        "cos_to_onpolicy_batch": cosine(g, g_on),
        "cos_to_reference": cosine(g, g_star),
    }
    for name, g in (("tokenwise", g_tok), ("score-function", g_pg), ("on-policy", g_on))
])
display(piece_df.round(4))

group_order, spans = parameter_spans(student)
layer_df = pd.DataFrame([
    {
        "group": key,
        "params": sum(hi - lo for lo, hi in spans[key]),
        "token_norm": take(g_tok, spans[key]).norm().item(),
        "pg_norm": take(g_pg, spans[key]).norm().item(),
        "onpolicy_norm": take(g_on, spans[key]).norm().item(),
        "cos_token_onpolicy": cosine(take(g_tok, spans[key]), take(g_on, spans[key])),
    }
    for key in group_order
])
display(layer_df.round(4))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(layer_df["group"], layer_df["cos_token_onpolicy"])
ax.set_xlabel("cosine(tokenwise, on-policy)")
ax.invert_yaxis()
ax.set_title(f"Gradient agreement by parameter group  N={N_ANAL}")
fig.tight_layout()
plt.show()

rewards, logp_a = sequence_rewards(student, teacher, letters, ALPHA)
weight = traj_w(rewards.detach())
params = student_params(student)
direct, score = [], []
for t in range(T):
    direct.append(flat_grad(torch.autograd.grad(rewards[:, t].mean(), params, retain_graph=True)))
    score.append(flat_grad(torch.autograd.grad((weight[:, t] * logp_a[:, t]).mean(), params, retain_graph=True)))
direct = torch.stack(direct)
score = torch.stack(score)
tok_err = (direct.sum(0) - g_tok).norm() / g_tok.norm()
on_err = (direct.sum(0) + score.sum(0) - g_on).norm() / g_on.norm()
print(f"position sum vs tokenwise rel={tok_err:.2e}  vs on-policy rel={on_err:.2e}")

cum_tok = torch.cumsum(direct, 0)
cum_full = torch.cumsum(direct + score, 0)
cos_tok_cum = [cosine(cum_tok[t], g_on) for t in range(T)]
cos_full_cum = [cosine(cum_full[t], g_on) for t in range(T)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ts = torch.arange(T)
axes[0].plot(ts, direct.norm(dim=1).cpu(), marker="o", ms=3, label=r"tokenwise $\nabla D_t$")
axes[0].plot(ts, score.norm(dim=1).cpu(), marker="o", ms=3, label=r"score $(R-b)\nabla\log\pi_t$")
axes[0].set_xlabel("position t")
axes[0].set_ylabel("gradient norm")
axes[0].legend()
axes[0].set_title("Per-position gradient norm")
axes[1].plot(ts, cos_tok_cum, marker="o", ms=3, label="cumsum tokenwise")
axes[1].plot(ts, cos_full_cum, marker="o", ms=3, label="cumsum on-policy")
axes[1].set_xlabel("last position included")
axes[1].set_ylabel("cosine to full on-policy grad")
axes[1].legend(fontsize=8)
axes[1].set_title("Cumulative gradient")
fig.tight_layout()
plt.show()

fd_letters = letters[:8]
g_fd = batch_grad(student, teacher, fd_letters, lambda r: torch.zeros_like(r), ALPHA)
coord = int(g_fd.abs().argmax())
offset = 0
param, index = None, None
for p in student_params(student):
    n = p.numel()
    if offset <= coord < offset + n:
        param, index = p, coord - offset
        break
    offset += n
eps = 1e-3
with torch.no_grad():
    orig = param.view(-1)[index].item()
    param.view(-1)[index] = orig + eps
loss_p = sequence_rewards(student, teacher, fd_letters, ALPHA)[0].sum(-1).mean()
with torch.no_grad():
    param.view(-1)[index] = orig - eps
loss_m = sequence_rewards(student, teacher, fd_letters, ALPHA)[0].sum(-1).mean()
with torch.no_grad():
    param.view(-1)[index] = orig
fd = (loss_p.detach().item() - loss_m.detach().item()) / (2 * eps)
rel = abs(g_fd[coord].item() - fd) / max(abs(fd), 1e-12)
print(f"finite difference  auto={g_fd[coord].item():.6e}  fd={fd:.6e}  rel={rel:.3e}")


## 3. Credit weights on one pool

Same \(W_t\) menu as `notebook/test.ipynb`. Each row is the mean of per-sequence
gradients on `N_METRICS` on-policy strings, compared with `g_star`.

`"no PG term"` is the tokenwise gradient. `"full trajectory (R - b)"` is the
estimator averaged into `g_star`. `"reward-to-go held-out baseline"` is its
reward-to-go form. At \(\gamma=1\), `G_t - batch_mean_t` and
`G(r - batch_mean_t)` are the same weight, and the `/remaining` pair matches too.


In [ ]:
weight_fns = {
    "full trajectory (R - b)": traj_w,
    "reward-to-go held-out baseline": rtg_w,
    **credit_weight_fns(T),
}
metric_letters = rollout(student, N_METRICS, T)

rows = []
for name, fn in weight_fns.items():
    g = per_sample_grads(student, teacher, metric_letters, fn, ALPHA)
    rows.append({"variant": name, **grad_metrics(g, g_star)})
credit_df = pd.DataFrame(rows)
display(credit_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
order = list(credit_df["variant"])
axes[0].barh(order, credit_df["snr"])
axes[0].set_xlabel("SNR")
axes[0].invert_yaxis()
axes[1].barh(order, credit_df["cos"])
axes[1].set_xlabel("cosine to on-policy reference")
axes[1].invert_yaxis()
fig.suptitle(f"Credit weights  N={N_METRICS}  T={T}  {divergence_name(ALPHA)}")
fig.tight_layout()
plt.show()


## 4. Batch-size sweep

Each point is the cosine of one micro-batch gradient to `g_star`, averaged over
`REPS` subsets of a fixed pool. The position-wise baseline inside `G_t - batch_mean_t`
is recomputed on that subset, as in a training step.


In [ ]:
pool = rollout(student, N_POOL, T)
sweep_rows = []
for B in BATCH_SIZES:
    print(f"B={B}", flush=True)
    acc = {name: [] for name in weight_fns}
    for _ in range(REPS):
        batch = pool[torch.randperm(N_POOL, device=pool.device)[:B]]
        for name, fn in weight_fns.items():
            g = batch_grad(student, teacher, batch, fn, ALPHA)
            acc[name].append(cosine(g, g_star))
    for name, vals in acc.items():
        v = torch.tensor(vals)
        sweep_rows.append({
            "B": B,
            "variant": name,
            "cos_mean": v.mean().item(),
            "cos_std": v.std(unbiased=False).item(),
        })

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df.pivot(index="B", columns="variant", values="cos_mean").reindex(columns=list(weight_fns)).round(3))

fig, ax = plt.subplots(figsize=(10.5, 4.8))
for name, sub in sweep_df.groupby("variant", sort=False):
    ax.plot(sub["B"], sub["cos_mean"], marker="o", label=name)
ax.set_xscale("log", base=2)
ax.set_xlabel("batch size B")
ax.set_ylabel("cosine to on-policy reference")
ax.set_title(
    f"Cosine vs B  T={T}  {divergence_name(ALPHA)}  reference split-half={ref_cos:.2f}"
)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
plt.show()
